# 02 — LSE & GCC Data Collection

Collects financial reports from London Stock Exchange (LSE) listed
companies and Gulf Cooperation Council (GCC) entities. IFRS-based
reports complement SEC EDGAR (US GAAP) data for training.

**Target**: ~100 LSE + 50 GCC reports

In [ ]:
# Cell 1: Setup
!pip install -q requests beautifulsoup4 lxml tqdm PyYAML PyMuPDF

import requests, json, time, os
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
LSE_DIR = DATA_DIR / 'lse'
GCC_DIR = DATA_DIR / 'gcc'
LSE_DIR.mkdir(parents=True, exist_ok=True)
GCC_DIR.mkdir(parents=True, exist_ok=True)

LSE_TARGET = cfg['data']['lse_target_reports']
GCC_TARGET = cfg['data']['gcc_target_reports']
HEADERS = {'User-Agent': 'Numera ML training (research)'}
print(f'Targets — LSE: {LSE_TARGET}, GCC: {GCC_TARGET}')

In [ ]:
# Cell 2: FTSE 100 company list
FTSE_COMPANIES = [
    {'ticker': 'SHEL', 'name': 'Shell plc'},
    {'ticker': 'AZN', 'name': 'AstraZeneca'},
    {'ticker': 'HSBA', 'name': 'HSBC Holdings'},
    {'ticker': 'ULVR', 'name': 'Unilever'},
    {'ticker': 'BP', 'name': 'BP plc'},
    {'ticker': 'GSK', 'name': 'GSK plc'},
    {'ticker': 'RIO', 'name': 'Rio Tinto'},
    {'ticker': 'BATS', 'name': 'British American Tobacco'},
    {'ticker': 'DGE', 'name': 'Diageo'},
    {'ticker': 'LSEG', 'name': 'London Stock Exchange Group'},
]

GCC_COMPANIES = [
    {'ticker': '2222', 'name': 'Saudi Aramco', 'exchange': 'Tadawul'},
    {'ticker': '1180', 'name': 'Al Rajhi Bank', 'exchange': 'Tadawul'},
    {'ticker': '2010', 'name': 'SABIC', 'exchange': 'Tadawul'},
    {'ticker': 'FAB', 'name': 'First Abu Dhabi Bank', 'exchange': 'ADX'},
    {'ticker': 'ADNOC', 'name': 'ADNOC Distribution', 'exchange': 'ADX'},
    {'ticker': 'EMAAR', 'name': 'Emaar Properties', 'exchange': 'DFM'},
    {'ticker': 'DIB', 'name': 'Dubai Islamic Bank', 'exchange': 'DFM'},
    {'ticker': 'QNBK', 'name': 'Qatar National Bank', 'exchange': 'QSE'},
]
print(f'FTSE: {len(FTSE_COMPANIES)}, GCC: {len(GCC_COMPANIES)}')

In [ ]:
# Cell 3: PDF download helpers
AR_SEARCH = 'https://www.annualreports.com/Company/{ticker}'

def find_pdf_links(url: str) -> list[str]:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code != 200:
            return []
        soup = BeautifulSoup(resp.text, 'lxml')
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            text = a.get_text(strip=True).lower()
            if href.endswith('.pdf') and any(kw in text for kw in ['annual', 'report', 'financial']):
                if href.startswith('/'):
                    from urllib.parse import urljoin
                    href = urljoin(url, href)
                links.append(href)
        return links[:3]
    except Exception as e:
        print(f'  Error: {e}')
        return []

def download_pdf(url: str, out_path: Path) -> bool:
    if out_path.exists():
        return True
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
        resp.raise_for_status()
        out_path.write_bytes(resp.content)
        return True
    except Exception:
        return False

In [ ]:
# Cell 4: Download LSE and GCC reports
lse_reports, gcc_reports = [], []

for co in tqdm(FTSE_COMPANIES[:LSE_TARGET], desc='LSE'):
    for i, link in enumerate(find_pdf_links(AR_SEARCH.format(ticker=co['ticker']))):
        out = LSE_DIR / f"{co['ticker']}_{i}.pdf"
        if download_pdf(link, out):
            lse_reports.append({**co, 'path': str(out)})
    time.sleep(1)

for co in tqdm(GCC_COMPANIES[:GCC_TARGET], desc='GCC'):
    for i, link in enumerate(find_pdf_links(AR_SEARCH.format(ticker=co['ticker']))):
        out = GCC_DIR / f"{co['exchange']}_{co['ticker']}_{i}.pdf"
        if download_pdf(link, out):
            gcc_reports.append({**co, 'path': str(out)})
    time.sleep(1)

print(f'LSE: {len(lse_reports)}, GCC: {len(gcc_reports)}')

In [ ]:
# Cell 5: Extract text and save metadata
import fitz

def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    try:
        doc = fitz.open(str(pdf_path))
        pages = [{'page': i+1, 'text': p.get_text().strip()} for i, p in enumerate(doc)]
        doc.close()
        return pages
    except Exception as e:
        print(f'  Error: {pdf_path}: {e}')
        return []

all_reports = lse_reports + gcc_reports
dataset = []
for r in tqdm(all_reports, desc='Extracting text'):
    pages = extract_pdf_pages(Path(r['path']))
    dataset.append({**r, 'pages': pages, 'total_pages': len(pages)})

meta_file = DATA_DIR / 'lse_gcc_metadata.json'
with open(meta_file, 'w') as f:
    json.dump(dataset, f, indent=2, default=str)

total_pages = sum(r['total_pages'] for r in dataset)
print(f'Total reports: {len(dataset)}, pages: {total_pages}')
print(f'Saved: {meta_file}')